## Automated LLM-Based Literature Extraction Pipeline Development

Lead     : `Aditya`

Issue    : [Github Issue #115](https://github.com/petadex/igem-toronto/issues/115) — _Automated LLM-Based Literature Extraction Pipeline Development

Start    : `2026-07-18`

Complete : `2026-10-10`

Files    : `~/igem-toronto/files/YYMMDD_issue106_<slug>/` — local working directory (active analysis, intermediate outputs)

S3 files :
- `s3://plastic-literature-activity-data/csvs/`
- `s3://plastic-literature-activity-data/extracted_activity_jsons/`
- `s3://plastic-literature-activity-data/docling_sectioned_text_jsons/`
- `s3://plastic-literature-activity-data/original_pdfs/`

---

### Data Accessed: 2026-10-10
```bash
```

## Introduction

Accurate and scalable aggregation of plastic-degrading enzyme activity data from academic literature is limited by the bottleneck of manual human curation found in resources like PAZY. To overcome this constraint, I developed an automated, high-precision extraction pipeline powered by large language models to parse thousands of academic PDFs. The system extracts activity measurements specifically for experimentally validated plastic-degrading enzymes while filtering out unverified targets.

The pipeline utilizes an iterative multi-pass extraction architecture and targeted LLM verification passes to eliminate hallucinations, enforce entity grounding and recover missing values. Synthetic controls and cross-model benchmarks were developed to quantify extraction precision, recall and relational integrity prior to full-scale deployment across literature corpora from the previous phase scraping.

## Objectives

1. Engineer a multi-stage LLM extraction and verification pipeline using Claude Sonnet on AWS Bedrock to extract enzyme and experimental assay parameters with verbatim evidence grounding.
2. Generate spiked synthetic controls within real paper to assess true positive capture rates, decoy rejection rates and relational pool linkage.
3. Benchmark LLM models that I have access to (Sonnet, Haiku, Opus and hybrid configurations of the three) against a cross-model consensus truth pool to determine optimal accuracy and cost.

---

## Materials and Methods

### System Initialization

The pipeline was orchestrated on AWS Bedrock utilizing Claude 4.6 Sonnet, Haiku and Opus models. Local and cloud compute environments integrated Python execution environments using regex modules, pandas and deterministic verification scripts to execute non-LLM validation and tabular cleaning steps.

### Data Initialization

```bash
# Accessed: 2026-10-10
# <command used to generate or download input data>
```

### Processing Steps

**Phase 1: Multi-Stage Extraction and Verification Pipeline Development**
- Pass 1: Sonnet 4.6 generates a dossier of all the plastic degrading enzymes in the paper and the experiments those enzymes participated with the corresponding substrates in using chronological thinking.
- Pass 2: Using the dossier from pass 1 Sonnet 4.6 extracts the relevant protein and activity data along with the exact verbatim evidence sentence the value was pulled from.
- Verification step 1: Using regex to check if the verbatim evidence sentences truly originated from the papers or were hallucinated.
- Verification step 2: Using simple python, the enzyme-substrate combinations from the dossier in pass 1 is compared to the extracted list in pass 2, any new proteins are flagged for review and any missing proteins are added in a notes section for LLM review.
- Verification step 3: A list of substrates is generated and compared against an exhaustive list of substrates to lightly flag any substrates not found in the list.
- Verification step 4: Sonnet 4.6 checks to see if proteins are actually tested on plastic substrates and removes any proteins that aren't with associated protein data.
- Rescue step 4.1: Sonnet 4.6 checks if any proteins that are tested against plastic/rubber substrates are not mentioned in the list and adds them if they are.
- Verification step 5: Sonnet 4.6 checks if the values are correct and originate from the evidence sentence, replacing any incorrect values with either the correct value or "Not_Found".
- Rescue step 5.1: Sonnet 4.6 looks for their actual values and an accompanying evidence sentence for "Not_Found" values and verification step 4 is rerun.
- Verification step 6: Sonnet 4.6 ensures that each "relational pool" is correctly associated, flagging any values that aren't.
- Rescue step 6.1: Sonnet 4.6 takes the flagged values and associates them with the correct relational pool.

**Phase 2: Synthetic Control Creation and Pipeline Fidelity Validation**
- Step 1: Inject synthetic protein entities, plausible activity measurements and non-degrading decoy proteins into 200 distinct academic papers.
- Step 2: Process the spiked corpus through pipeline iterations (v1 baseline through final v16 multi-step rescue pipeline).
- Step 3: Quantify true positive capture rates, decoy false-positive acceptance rates and relational integrity linking assays to specific enzymes.

**Phase 3: Model Benchmarking**
- Step 1: Evaluate extraction performance across Sonnet, Haiku, Opus and hybrid passes against a core truth pool (N=821 proteins).
- Step 2: Compare verbatim evidence sentence overlap and semantic concordance across models.

**Phase 4: Production Corpus Execution**
- Deploy the finalized pipeline across ~4,000 relevant papers and execute post-extraction data cleaning (collapsing duplicate entries, filtering non-numeric text and flagging low-confidence tiers).

```bash
# Uploaded: 2026-10-10
aws s3 cp <output_file> s3://plastic-literature-activity-data/<path>/
```

## Results & Discussion

The finalized v16 extraction architecture operates across 8 LLM steps and 3 non-LLM steps at an approximate unit cost of $0.33 per paper. Across synthetic spike-in evaluations, the pipeline achieved zero decoy contamination while maintaining near-complete capture of validated entities. Model benchmarking demonstrated that Sonnet provided the optimal balance of recall, precision and global metric accuracy, whereas Opus suffered from severe under-extraction (9.1% recall). Production execution across 4,049 papers yielded 51,455 high-quality measurements across diverse polymer families.

**Key metric**: 40,803 / 51,455 retained measurements (79%) _achieved high-confidence classification (retaining numeric value, verified unit and identified enzyme/substrate pair)_.

**Breakdown by category**:
**Synthetic Controls**
- 100% decoy rejection rate (no false positives)
- 99.3% (152/153) Valid Spike-in Capture Rate (true positives)
- 100% (152/152) Relational linkage Assays and Proteins
- 98.8% (712/720) Assay Variable Extraction Rate
- 98.0% (196/200) Activity Metric Extraction Rate

Controls 2.svg

**Cross-Model Benchmark Performance (N=821 consensus proteins)**
- Sonnet: 95.0% Precision, 87.1% Recall, 90.9% F1 Score and 97.7% Global Metric Accuracy
- Haiku: 90.1% Precision, 81.5% Recall, 85.6% F1 Score and 94.4% Global Metric Accuracy
- Opus: 100.0% Precision, 9.1% Recall, 16.7% F1 Score and 98.1% Global Metric Accuracy
- Opus -> Sonnet Hybrid: 92.9% Precision, 80.0% Recall, 86.0% F1 Score and 98.5% Global Metric Accuracy
- Opus -> Haiku Hybrid: 96.2% Precision, 76.7% Recall, 85.4% F1 Score and 95.4% Global Metric Accuracy

**Production Corpus Overview (4049 papers)**
- Cleaned Entities: 27,244 experiments, 31,441 enzymes (after deduplicating 419 records) and 51,455 cleaned measurement rows (from 61,771 raw rows)
- Primary Removal Drivers: 4,508 duplicate rows collapsed, 2,993 lacking measurement values, 2,354 descriptive texts, 325 missing enzymes, 111 implausible units and 25 implausible temperatures
- Coverage Distribution: Median 5 experiments per paper (max 80), median 10 retained measurements per paper and median 3 enzymes per paper
- Polymer Representation: PET & related polyesters (16,509 measurements; 32%), aliphatic polyesters (PCL/PLA/PHA/PBS; 15,026 measurements; 29%), polyolefins (3,429 measurements; 7%), polyurethane (2,698 measurements; 5%) and polystyrene (2,139 measurements; 4%)
- Substrate Physical Form: Film (11,438), powder (3,553), solution (1,500), emulsion (963), granules (549) and nanoparticles (536)

**Observations**:
- Pairwise model comparisons revealed 100% verbatim evidence sentence correlation.
- When different models isolated the same protein entity, their evidence spans and extracted values showed zero semantic contradiction.
- Strict extraction prompting prioritized precision over recall, deliberately omitting ambiguous values to prevent synthetic hallucination.
- Field completeness varied across experimental parameters: enzyme (100%) and substrate (99%) identification were near-universal, while assay environment parameters such as pH (40%), buffer composition (39%) and expression plasmid (8%) were less frequently reported or captured.
- Models rarely hallucinate past the verification steps.

**Follow-up questions**:
- How will ingesting unstructured supplementary tables and files impact recall rates across buffer and plasmid fields?
- Will increasing input figure resolutions above 512x512 improve extraction of kinetic metrics presented strictly in graphical formats?
- What downstream gains in precision can be achieved by transferring unit standardization and mathematical conversions entirely to native Python modules?
- How well do the models actually compare against human data and how can they be improved?